In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter

import os

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

# plt.rcParams.update({
#     'font.family': 'serif',
#     'font.serif':  ['Times New Roman'],
#     'mathtext.fontset': 'stix',
# })

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif':  ['DejaVu Serif'],
    'mathtext.fontset': 'stix',
})

from namespaces import DA

os.chdir('/home/mai/notebooks/final_thesis/')
os.getcwd()

'/home/mai/notebooks/final_thesis'

In [2]:
data = np.load(os.path.join(DA.paths.data, 'dgraphfin.npz'))
print(f'Keys in .npz: {list(data.keys())}')

x = data['x'].astype(np.float64)
y = data['y']
 
N, D = x.shape
print(f'Nodes: {N:,} | Features: {D}')
 
fraud_mask  = (y == 1)
benign_mask = (y == 0)
 
n_fraud  = fraud_mask.sum()
n_benign = benign_mask.sum()
print(f'Fraud: {n_fraud:,} | Benign: {n_benign:,}')

Keys in .npz: ['x', 'y', 'edge_index', 'edge_type', 'edge_timestamp', 'train_mask', 'valid_mask', 'test_mask']
Nodes: 3,700,550 | Features: 17
Fraud: 15,509 | Benign: 1,210,092


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Compute per-feature missing rates
# ─────────────────────────────────────────────────────────────────────────────
 
# Missing = NaN OR equal to missing_val
MISSING_IND = -1

missing_mask = np.isnan(x) | (x == MISSING_IND)
missing_rate_all    = missing_mask.mean(axis=0)                      # [D]
missing_rate_fraud  = missing_mask[fraud_mask].mean(axis=0)          # [D]
missing_rate_benign = missing_mask[benign_mask].mean(axis=0)         # [D]
 
print(f'\nOverall missing rate: {missing_mask.mean():.2%}')
print(f'Per-feature missing rates:')
for i in range(D):
    print(f'  Feature {i:>2d}: {missing_rate_all[i]:.2%} '
          f'(fraud: {missing_rate_fraud[i]:.2%}, '
          f'benign: {missing_rate_benign[i]:.2%})')


Overall missing rate: 49.95%
Per-feature missing rates:
  Feature  0: 14.72% (fraud: 0.01%, benign: 0.21%)
  Feature  1: 14.77% (fraud: 0.01%, benign: 0.21%)
  Feature  2: 54.17% (fraud: 65.16%, benign: 35.23%)
  Feature  3: 54.21% (fraud: 65.18%, benign: 35.27%)
  Feature  4: 54.21% (fraud: 65.18%, benign: 35.27%)
  Feature  5: 57.98% (fraud: 68.72%, benign: 38.51%)
  Feature  6: 54.17% (fraud: 65.16%, benign: 35.23%)
  Feature  7: 57.98% (fraud: 68.72%, benign: 38.51%)
  Feature  8: 54.21% (fraud: 65.18%, benign: 35.27%)
  Feature  9: 54.21% (fraud: 65.18%, benign: 35.27%)
  Feature 10: 0.00% (fraud: 0.00%, benign: 0.00%)
  Feature 11: 58.00% (fraud: 69.68%, benign: 39.25%)
  Feature 12: 58.00% (fraud: 69.68%, benign: 39.25%)
  Feature 13: 57.98% (fraud: 68.72%, benign: 38.51%)
  Feature 14: 60.05% (fraud: 70.53%, benign: 40.76%)
  Feature 15: 72.26% (fraud: 78.34%, benign: 55.86%)
  Feature 16: 72.26% (fraud: 78.34%, benign: 55.86%)


In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # Plot 1: Missing value rates per feature
# # ─────────────────────────────────────────────────────────────────────────────

# PLOT_DIR = './plots'
# os.makedirs(PLOT_DIR, exist_ok=True)
 
# fig_miss, ax_miss = plt.subplots(figsize=(max(D * 0.5, 8), 4))
 
# bar_width = 0.35
# feat_idx  = np.arange(D)
 
# ax_miss.bar(feat_idx - bar_width/2, missing_rate_fraud * 100,
#             bar_width, label='Fraud', color='#d62728', alpha=0.8)
# ax_miss.bar(feat_idx + bar_width/2, missing_rate_benign * 100,
#             bar_width, label='Benign', color='#1f77b4', alpha=0.8)
 
# ax_miss.set_xlabel('Feature Index')
# ax_miss.set_ylabel('Missing Rate (%)')
# # ax_miss.set_title('Missing Value Rate per Feature (Fraud vs Benign)')
# ax_miss.set_xticks(feat_idx)
# ax_miss.set_xticklabels([str(i) for i in range(D)], fontsize=7)
# ax_miss.legend()
# ax_miss.grid(axis='y', alpha=0.3)
 
# fig_miss.tight_layout()
# # plt.show()
# path_miss = os.path.join(PLOT_DIR, 'feature_missing_rates.png')
# fig_miss.savefig(path_miss, dpi=300, bbox_inches='tight')
# print(f'\nSaved: {path_miss}')
# plt.close(fig_miss)


Saved: ./plots/feature_missing_rates.png


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# Feature distributions — fraud vs benign vs background
# ─────────────────────────────────────────────────────────────────────────────
PLOT_DIR = './plots'
os.makedirs(PLOT_DIR, exist_ok=True)

N_COLS = 4
FIG_SCALE = 3.0
BINS = 50


fraud_mask  = (y == 1)
benign_mask = (y == 0)
bg_mask = (y==2) | (y==3)

classes = [
    ('Benign',     benign_mask, '#1f77b4'),  # blue
    ('Fraud',      fraud_mask,  '#ff7f0e'),  # orange
    ('Background', bg_mask,     '#9467bd'),  # purple
]

n_cols = N_COLS
n_rows = int(np.ceil(D / n_cols))
fig_size = (FIG_SCALE * n_cols, FIG_SCALE * n_rows + 0.8)

fig, axes = plt.subplots(n_rows, n_cols, figsize=fig_size)
axes = axes.flatten()

for i in range(D):
    ax = axes[i]

    col = x[:, i]
    valid_mask = ~missing_mask[:, i]

    all_valid = col[valid_mask]
    if len(all_valid) == 0:
        ax.text(0.5, 0.5, 'All missing', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='gray')
        ax.set_title(f'Feature {i}', fontsize=13)
        continue

    lo, hi = np.percentile(all_valid, [1, 99])
    bins = np.linspace(lo, hi, BINS + 1)

    for name, mask, color in classes:
        vals = col[mask & valid_mask]
        if len(vals) > 0:
            if name == 'Background':
                ax.hist(np.clip(vals, lo, hi), bins=bins, density=True,
                        histtype='step', linewidth=1.0,
                        color=color, label=name)
            else:
                ax.hist(np.clip(vals, lo, hi), bins=bins, density=True,
                        alpha=0.4, color=color, label=name)

    ax.set_title(f'Feature {i}', fontsize=13)
    ax.tick_params(labelsize=10)

    miss_f  = missing_mask[fraud_mask, i].mean()
    miss_b  = missing_mask[benign_mask, i].mean()
    miss_bg = missing_mask[bg_mask, i].mean()
    ax.text(0.97, 0.95,
            f'miss: F={miss_f:.1%}\n'
            f'      B={miss_b:.1%}\n'
            f'         BG={miss_bg:.1%}',
            transform=ax.transAxes, fontsize=9, ha='right', va='top')
    
    if i % n_cols == 0:
        ax.set_ylabel('Density', fontsize=11)

# Hide unused subplots
for i in range(D, len(axes)):
    axes[i].set_visible(False)

# Shared legend at bottom center
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels,
           loc='lower center',
           ncol=3,
           fontsize=12,
           frameon=False,
           bbox_to_anchor=(0.5, -0.01))

fig.tight_layout()
fig.subplots_adjust(bottom=0.05)

path_dist = os.path.join(PLOT_DIR, 'feature_distributions_all_classes.png')
fig.savefig(path_dist, dpi=300, bbox_inches='tight')
print(f'Saved: {path_dist}')
plt.close(fig)

Saved: ./plots/feature_distributions_all_classes.png
